In [2]:
# 1. 导入必要的库
from qiskit_aer import Aer
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit.circuit.library import QFTGate, UnitaryGate
import numpy as np
from scipy.linalg import expm
import matplotlib.pyplot as plt

# ==========================================
# 修改配置：
# n_c = 6 (6个时钟量子比特)
# ==========================================
n_b = 4  
n_c = 6  # 修改为 6
n_a = 1  

# 2. 定义矩阵 A 和向量 b (用于构建电路结构)
N = 16
matrix_A = np.zeros((N, N))
# 简单的对角矩阵占位，确保数学计算不报错
np.fill_diagonal(matrix_A, 1) 

vector_b = np.array([0.2625, 0.4375, 0.35, 0.175, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
normalized_b = vector_b / np.linalg.norm(vector_b)

# 计算 t
eigenvalues = np.linalg.eigvalsh(matrix_A)
lambda_max_abs = np.max(np.abs(eigenvalues))
t = np.pi / lambda_max_abs

# 3. 辅助函数：构建旋转层模块
def create_rotation_gate(n_c, t, C=0.25):
    """
    封装旋转层，避免绘制数千个门导致内存溢出。
    """
    qr_clock = QuantumRegister(n_c, 'clock')
    qr_ancilla = QuantumRegister(1, 'ancilla')
    qc_rot = QuantumCircuit(qr_clock, qr_ancilla)
    
    # 哪怕是 n_c=6 (63次循环)，在封装模式下也就是生成一个黑盒指令，
    # 绘图时不会展开内部，所以是安全的。
    for k in range(1, 2**n_c):
        if k < 2**(n_c - 1): k_signed = k 
        else: k_signed = k - 2**n_c 
        
        lambda_val = (2 * np.pi * k_signed) / (t * (2**n_c))
        
        if abs(lambda_val) < 1e-6: continue
        
        ratio = C / lambda_val
        if ratio > 1: ratio = 1
        if ratio < -1: ratio = -1
        theta = 2 * np.arcsin(ratio)
        
        k_binary = format(k, f'0{n_c}b')[::-1]
        
        for idx, bit in enumerate(k_binary):
            if bit == '0': qc_rot.x(qr_clock[idx])
                
        qc_rot.mcry(theta, qr_clock[:], qr_ancilla[0])
        
        for idx, bit in enumerate(k_binary):
            if bit == '0': qc_rot.x(qr_clock[idx])
            
    rot_inst = qc_rot.to_instruction(label="Eigenvalue Rotation")
    return rot_inst

# 4. 构建受控酉算子
def apply_controlled_u(qc, clock_qubits, input_qubits, matrix_A, t):
    for k in range(len(clock_qubits)):
        power = 2**k
        # 使用占位 Unitary 简化绘图
        u_gate = UnitaryGate(np.eye(2**len(input_qubits)), label=f"U^{power}")
        c_u_gate = u_gate.control(1)
        qc.append(c_u_gate, [clock_qubits[k]] + input_qubits[:])

def apply_inverse_controlled_u(qc, clock_qubits, input_qubits, matrix_A, t):
    for k in reversed(range(len(clock_qubits))):
        power = 2**k
        u_gate = UnitaryGate(np.eye(2**len(input_qubits)), label=f"U_dag^{power}")
        c_u_gate = u_gate.control(1)
        qc.append(c_u_gate, [clock_qubits[k]] + input_qubits[:])

# 5. 构建主电路
clock = QuantumRegister(n_c, name='clock')
input_reg = QuantumRegister(n_b, name='b')
ancilla = QuantumRegister(n_a, name='ancilla')
measurement = ClassicalRegister(n_b, name='c')
meas_ancilla = ClassicalRegister(1, name='ma')

circuit = QuantumCircuit(ancilla, clock, input_reg, measurement, meas_ancilla)

# (A) 状态准备
circuit.initialize(normalized_b, input_reg)
circuit.barrier()

# (B) QPE
circuit.h(clock)
apply_controlled_u(circuit, clock, list(input_reg), matrix_A, t)
circuit.barrier()

# (C) Inverse QFT
iqft = QFTGate(num_qubits=n_c).inverse()
iqft.label = "IQFT" 
circuit.append(iqft, clock)
circuit.barrier()

# (D) 旋转 (封装版)
rot_gate = create_rotation_gate(n_c, t)
circuit.append(rot_gate, list(clock) + list(ancilla))
circuit.barrier()

# (E) Uncompute QPE
qft = QFTGate(num_qubits=n_c)
qft.label = "QFT"
circuit.append(qft, clock)
circuit.barrier()

apply_inverse_controlled_u(circuit, clock, list(input_reg), matrix_A, t)

# === 关键步骤：在测量前应用 H 门 ===
# 这里的 H 门是 QPE 逆过程的最后一步（对应 QPE 开始时的 H 门）
circuit.h(clock)
circuit.barrier()

# (F) 测量
circuit.measure(ancilla, meas_ancilla)
circuit.measure(input_reg, measurement)

# === 绘制并保存电路图 ===
print("正在生成电路图 (n_c=6)...")

# scale: 稍微调小一点，因为 6+4+1=11 个比特会让图很高
# fold: 设为 50，让图稍微宽一点，避免换行太频繁
circuit.draw(output='mpl', filename='hhl_circuit_6qubits.png', style='iqp', scale=0.7, fold=50)

print("电路图已保存为 'hhl_circuit_6qubits.png'。")

正在生成电路图 (n_c=6)...
电路图已保存为 'hhl_circuit_6qubits.png'。
